# demo - the job the BRIDGE web app runs

The web app's **Run Model** button triggers the Databricks job `MDR_JOB_ID`, which runs this
notebook. It only reads the job parameters and hands them to `mmm.app_job.run_app_job`; all the
logic, and its tests, live there.

**Job parameters** (define them on the job; all blank by default):

| parameter | meaning |
|---|---|
| `config_file` | the config YAML the app uploaded to `Secondary Modelling/Config/` (blank = this folder's `config.yaml`) |
| `data_file` | the datacube in `Secondary Modelling/Data/` |
| `prior_file` | the feature-prior CSV in `Secondary Modelling/Prior/` |
| `mapping_file` | optional, in `Secondary Modelling/Mapping/` |
| `share_file` | optional, in `Secondary Modelling/Share/` |
| `run_id` | set it to `{{job.run_id}}`: it names the output folder `Outputs/<run_id>` |
| `base_path` | blank = `/dbfs/mnt/testuat/Secondary Modelling` (a `/Volumes/...` path works too) |

Only the input paths, the output folder and the run name are changed; every other setting runs
exactly as uploaded. This folder's `config.yaml` is never edited.

**Running it by hand:** leave every widget blank. It runs this folder's `config.yaml` as it stands,
like `run_real_data.py`, and publishes to `Outputs/manual_<time>`.

The synthetic parameter-recovery check that used to be this notebook's only cell is
`python synthetic_example.py`.

In [ ]:
import json
import os
import sys

# this folder holds the mmm package; make sure it is importable
sys.path.insert(0, os.getcwd())
from mmm.app_job import PARAMS, run_app_job

# Job parameters arrive as widgets. Defining them blank first lets the notebook
# also run by hand. The run id is the job parameter run_id = {{job.run_id}}:
# context.tags() is not available on this compute (it raised a Py4JError).
for name in PARAMS:
    dbutils.widgets.text(name, "")
params = {name: dbutils.widgets.get(name).strip() for name in PARAMS}
print(params)

In [ ]:
# Fits the model and publishes Outputs/<run_id>. A failed run is published
# first (so its 00_warnings can be read) and then re-raised, so the job FAILS
# with the real error - which the web app shows in its status popup.
info = run_app_job(params)
print(json.dumps(info, indent=2, default=str))

In [ ]:
# the web app reads this through the Jobs API (runs/get-output)
dbutils.notebook.exit(json.dumps(
    {k: info.get(k) for k in ("status", "run_id", "codebase", "output_dir", "seconds")},
    default=str))